# Yahtzee Simulation
Monte Carlo simulation of Yahtzee with multiple strategies. Each strategy plays thousands of full games; we compare average scores and score distributions.

In [1]:
import random
from collections import Counter
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

## Scoring Logic
All 13 Yahtzee categories plus bonus detection.

In [2]:
CATEGORIES = [
    "ones", "twos", "threes", "fours", "fives", "sixes",
    "three_of_a_kind", "four_of_a_kind", "full_house",
    "small_straight", "large_straight", "yahtzee", "chance"
]

def score_dice(dice, category):
    """Return the score for a given dice roll in a given category."""
    counts = Counter(dice)
    face_counts = sorted(counts.values(), reverse=True)
    sorted_unique = sorted(set(dice))

    if category == "ones":   return sum(d for d in dice if d == 1)
    if category == "twos":   return sum(d for d in dice if d == 2)
    if category == "threes": return sum(d for d in dice if d == 3)
    if category == "fours":  return sum(d for d in dice if d == 4)
    if category == "fives":  return sum(d for d in dice if d == 5)
    if category == "sixes":  return sum(d for d in dice if d == 6)

    if category == "three_of_a_kind":
        return sum(dice) if face_counts[0] >= 3 else 0
    if category == "four_of_a_kind":
        return sum(dice) if face_counts[0] >= 4 else 0
    if category == "full_house":
        return 25 if sorted(face_counts) == [2, 3] else 0
    if category == "small_straight":
        runs = [set(range(i, i+4)) for i in range(1, 4)]
        return 30 if any(r.issubset(set(dice)) for r in runs) else 0
    if category == "large_straight":
        return 40 if set(dice) in ({1,2,3,4,5}, {2,3,4,5,6}) else 0
    if category == "yahtzee":
        return 50 if face_counts[0] == 5 else 0
    if category == "chance":
        return sum(dice)
    raise ValueError(f"Unknown category: {category}")


def upper_bonus(scorecard):
    """35-point bonus if upper section sums to 63+."""
    upper = ["ones","twos","threes","fours","fives","sixes"]
    total = sum(scorecard[c] for c in upper if scorecard[c] is not None)
    filled = all(scorecard[c] is not None for c in upper)
    return 35 if filled and total >= 63 else 0


def total_score(scorecard):
    base = sum(v for v in scorecard.values() if v is not None)
    return base + upper_bonus(scorecard)

## Game Engine
Roll dice, keep a subset, re-roll — up to 3 rolls per turn.

In [3]:
def roll(n=5):
    return [random.randint(1, 6) for _ in range(n)]


def reroll(dice, keep_indices):
    """Keep dice at keep_indices, re-roll the rest."""
    kept = [dice[i] for i in keep_indices]
    new = roll(5 - len(kept))
    # Reassemble in original slot order so indexing stays stable
    result = list(dice)
    reroll_slots = [i for i in range(5) if i not in keep_indices]
    for slot, val in zip(reroll_slots, new):
        result[slot] = val
    return result


def play_turn(strategy, scorecard):
    """
    Play one turn using `strategy`.
    strategy(dice, roll_num, scorecard) -> (keep_indices, category_or_None)
      - If category is not None, score that category immediately (no more rolls).
      - keep_indices is used for the re-roll if rolling again.
    Returns (final_dice, chosen_category).
    """
    dice = roll()
    chosen_category = None

    for roll_num in range(1, 4):  # rolls 1, 2, 3
        keep_indices, chosen_category = strategy(dice, roll_num, scorecard)
        if chosen_category is not None or roll_num == 3:
            break
        dice = reroll(dice, keep_indices)

    # If strategy never picked a category (shouldn't happen), fall back to chance
    if chosen_category is None:
        open_cats = [c for c in CATEGORIES if scorecard[c] is None]
        chosen_category = open_cats[0]

    return dice, chosen_category


def play_game(strategy):
    """Play a full 13-turn game. Returns the final scorecard and total score."""
    scorecard = {c: None for c in CATEGORIES}
    for _ in range(13):
        dice, category = play_turn(strategy, scorecard)
        scorecard[category] = score_dice(dice, category)
    return scorecard, total_score(scorecard)

## Strategies

Four strategies to compare:
1. **Random** — keep random dice, score a random open category on roll 3
2. **Greedy** — always score the open category with the highest *current* value
3. **Yahtzee Hunter** — always try to build toward Yahtzee; fall back to greedy
4. **Expected Value** — on each re-roll, keep the subset that maximizes expected score across all open categories

In [4]:
def open_categories(scorecard):
    return [c for c in CATEGORIES if scorecard[c] is None]


# ── Strategy 1: Random ────────────────────────────────────────────────────────
def strategy_random(dice, roll_num, scorecard):
    if roll_num == 3:
        cat = random.choice(open_categories(scorecard))
        return list(range(5)), cat
    keep = random.sample(range(5), random.randint(0, 5))
    return keep, None


# ── Strategy 2: Greedy ────────────────────────────────────────────────────────
def best_category(dice, scorecard):
    open_cats = open_categories(scorecard)
    return max(open_cats, key=lambda c: score_dice(dice, c))

def strategy_greedy(dice, roll_num, scorecard):
    cat = best_category(dice, scorecard)
    score = score_dice(dice, cat)

    if roll_num == 3 or score > 0:
        return list(range(5)), cat

    counts = Counter(dice)
    max_count = max(counts.values())
    keep_face = max(f for f, c in counts.items() if c == max_count)
    keep = [i for i, d in enumerate(dice) if d == keep_face]
    return keep, None


# ── Strategy 3: Yahtzee Hunter ────────────────────────────────────────────────
def strategy_yahtzee_hunter(dice, roll_num, scorecard):
    counts = Counter(dice)
    max_count = max(counts.values())
    modal_face = max(counts, key=counts.get)
    keep = [i for i, d in enumerate(dice) if d == modal_face]

    if max_count == 5:
        cat = "yahtzee" if scorecard["yahtzee"] is None else best_category(dice, scorecard)
        return list(range(5)), cat

    if roll_num == 3:
        cat = best_category(dice, scorecard)
        return list(range(5)), cat

    return keep, None


# ── Strategy 4: Expected Value ────────────────────────────────────────────────
EV_SAMPLES     = 200
EV_SUB_SAMPLES = 20

def smart_keeps(dice, scorecard):
    """
    Generate only keeps that build toward an open category.
    Replaces the naive 2^5=32 enumeration with ~8-14 purposeful candidates.
    """
    open_cats = open_categories(scorecard)
    candidates = set()
    counts = Counter(dice)

    candidates.add(())              # re-roll everything
    candidates.add(tuple(range(5))) # keep everything

    # Upper section: keep 1..n dice of the relevant face value
    face_for_cat = {"ones":1,"twos":2,"threes":3,"fours":4,"fives":5,"sixes":6}
    for cat, face in face_for_cat.items():
        if cat in open_cats and counts[face] > 0:
            indices = [i for i, d in enumerate(dice) if d == face]
            for n in range(1, len(indices) + 1):
                candidates.add(tuple(indices[:n]))

    # Multiples: keep 2+ of any face (3-of-a-kind, 4-of-a-kind, Yahtzee)
    if {"three_of_a_kind","four_of_a_kind","yahtzee"} & set(open_cats):
        for face, cnt in counts.items():
            if cnt >= 2:
                indices = [i for i, d in enumerate(dice) if d == face]
                for n in range(2, cnt + 1):
                    candidates.add(tuple(indices[:n]))

    # Full house: keep an existing pair, triple, or pair+triple of different faces
    if "full_house" in open_cats:
        pairs = {f: [i for i,d in enumerate(dice) if d==f] for f,c in counts.items() if c >= 2}
        trips = {f: [i for i,d in enumerate(dice) if d==f] for f,c in counts.items() if c >= 3}
        for idxs in pairs.values():
            candidates.add(tuple(idxs[:2]))
        for idxs in trips.values():
            candidates.add(tuple(idxs[:3]))
        for pf, pi in pairs.items():
            for tf, ti in trips.items():
                if pf != tf:
                    candidates.add(tuple(sorted(pi[:2] + ti[:3])))

    # Straights: keep the longest consecutive run of unique values (length >= 3)
    if {"small_straight","large_straight"} & set(open_cats):
        unique_vals = sorted(set(dice))
        run = [unique_vals[0]]
        for v in unique_vals[1:]:
            if v == run[-1] + 1:
                run.append(v)
            else:
                if len(run) >= 3:
                    idx = tuple(next(i for i,d in enumerate(dice) if d==v2) for v2 in run)
                    candidates.add(tuple(sorted(idx)))
                run = [v]
        if len(run) >= 3:
            idx = tuple(next(i for i,d in enumerate(dice) if d==v) for v in run)
            candidates.add(tuple(sorted(idx)))

    return [list(k) for k in candidates]


def expected_score_for_keep(keep_indices, dice, scorecard, remaining_rolls, n_samples=EV_SAMPLES):
    open_cats = open_categories(scorecard)
    if not open_cats:
        return 0

    total = 0
    for _ in range(n_samples):
        sim_dice = reroll(dice, keep_indices)
        if remaining_rolls > 1:
            best_val = 0
            for sub_keep in smart_keeps(sim_dice, scorecard):
                sub_ev = expected_score_for_keep(
                    sub_keep, sim_dice, scorecard, remaining_rolls - 1,
                    n_samples=EV_SUB_SAMPLES
                )
                best_val = max(best_val, sub_ev)
            total += best_val
        else:
            total += max(score_dice(sim_dice, c) for c in open_cats)
    return total / n_samples


def strategy_ev(dice, roll_num, scorecard):
    open_cats = open_categories(scorecard)
    remaining_rolls = 3 - roll_num

    if remaining_rolls == 0:
        cat = max(open_cats, key=lambda c: score_dice(dice, c))
        return list(range(5)), cat

    best_keep, best_val = list(range(5)), -1
    for keep in smart_keeps(dice, scorecard):
        ev = expected_score_for_keep(keep, dice, scorecard, remaining_rolls)
        if ev > best_val:
            best_val = ev
            best_keep = keep

    return best_keep, None

## Validation
Spot-check scoring rules and confirm each strategy can complete a full game.

In [5]:
# ── score_dice ────────────────────────────────────────────────────────────────
assert score_dice([1,1,2,3,4], "ones")            == 2
assert score_dice([6,6,6,6,6], "sixes")           == 30
assert score_dice([3,3,3,4,5], "three_of_a_kind") == 18
assert score_dice([3,3,3,3,5], "four_of_a_kind")  == 17
assert score_dice([3,3,3,4,5], "four_of_a_kind")  == 0   # only 3-of-a-kind
assert score_dice([1,1,2,2,2], "full_house")      == 25
assert score_dice([3,3,3,3,3], "full_house")      == 0   # yahtzee ≠ full house
assert score_dice([1,2,3,4,6], "small_straight")  == 30
assert score_dice([3,4,5,6,6], "small_straight")  == 30
assert score_dice([1,2,3,4,6], "large_straight")  == 0
assert score_dice([1,2,3,4,5], "large_straight")  == 40
assert score_dice([2,3,4,5,6], "large_straight")  == 40
assert score_dice([4,4,4,4,4], "yahtzee")         == 50
assert score_dice([4,4,4,4,5], "yahtzee")         == 0
assert score_dice([1,2,3,4,5], "chance")          == 15

# ── upper bonus ───────────────────────────────────────────────────────────────
sc_full = {c: None for c in CATEGORIES}
for face, cat in zip([5,10,15,20,25,30], ["ones","twos","threes","fours","fives","sixes"]):
    sc_full[cat] = face   # sum = 105, qualifies for +35
assert upper_bonus(sc_full) == 35

sc_low = {c: None for c in CATEGORIES}
for face, cat in zip([1,2,3,4,5,6], ["ones","twos","threes","fours","fives","sixes"]):
    sc_low[cat] = face    # sum = 21, no bonus
assert upper_bonus(sc_low) == 0

# ── smoke test: one game per non-EV strategy ──────────────────────────────────
for strat in [strategy_random, strategy_greedy, strategy_yahtzee_hunter]:
    sc, score = play_game(strat)
    assert len(sc) == 13, "scorecard must have 13 categories"
    assert all(v is not None for v in sc.values()), "all categories must be filled"
    assert 0 <= score, "score must be non-negative"
    print(f"{strat.__name__:<30}  score = {score}")

# ── EV smoke test: 1 game with current EV_SAMPLES ────────────────────────────
import time
t0 = time.perf_counter()
sc_ev, score_ev = play_game(strategy_ev)
dt = time.perf_counter() - t0
assert all(v is not None for v in sc_ev.values())
print(f"{'strategy_ev':<30}  score = {score_ev}  ({dt:.1f}s per game)")
print(f"\nAll assertions passed. At {dt:.1f}s/game × 500 games / 16 cores ≈ {dt*500/16:.0f}s total.")

strategy_random                 score = 47
strategy_greedy                 score = 154
strategy_yahtzee_hunter         score = 175
strategy_ev                     score = 218  (23.0s per game)

All assertions passed. At 23.0s/game × 500 games / 16 cores ≈ 720s total.


## Run the Simulation
10,000 games per strategy (EV strategy uses fewer due to runtime).

In [ ]:
# def simulate(strategy, n_games):
#     return [play_game(strategy)[1] for _ in range(n_games)]

# N = 10_000
# N_EV = 500  # EV is slower due to nested Monte Carlo

# print("Running Random...")
# scores_random = simulate(strategy_random, N)
# print("Running Greedy...")
# scores_greedy = simulate(strategy_greedy, N)
# print("Running Yahtzee Hunter...")
# scores_yahtzee = simulate(strategy_yahtzee_hunter, N)
# print(f"Running EV ({N_EV} games)...")
# scores_ev = simulate(strategy_ev, N_EV)
# print("Done.")

## Parallel Simulation
`joblib` with `loky` backend splits games across all 16 physical cores of the 9800X3D.
Each worker gets its own RNG seed so results are statistically independent.
We use physical cores (16), not logical (32) — the workload is compute-bound so hyperthreading doesn't help.

In [6]:
# pip install joblib  (usually already present via scikit-learn/numpy ecosystem)
from joblib import Parallel, delayed
import time

N_CORES = 16  # 9800X3D physical cores

# loky (joblib's default) uses cloudpickle, so notebook-defined functions are picklable
STRATEGY_MAP = {
    "random":         strategy_random,
    "greedy":         strategy_greedy,
    "yahtzee_hunter": strategy_yahtzee_hunter,
    "ev":             strategy_ev,
}

def _run_chunk(strategy_name, n_games, seed):
    """Top-level worker function — each process gets its own RNG state."""
    import random as _random
    _random.seed(seed)
    strategy = STRATEGY_MAP[strategy_name]
    return [play_game(strategy)[1] for _ in range(n_games)]


def simulate_parallel(strategy_name, n_games, n_jobs=N_CORES):
    chunk_sizes = [n_games // n_jobs] * n_jobs
    for i in range(n_games % n_jobs):
        chunk_sizes[i] += 1
    seeds = [random.randint(0, 2**31) for _ in range(n_jobs)]

    chunks = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_run_chunk)(strategy_name, c, s)
        for c, s in zip(chunk_sizes, seeds)
    )
    return [score for chunk in chunks for score in chunk]

In [7]:
from tqdm.notebook import tqdm  # renders as a widget in Jupyter

def simulate_parallel_progress(strategy_name, n_games, n_jobs=N_CORES, games_per_chunk=25):
    """
    Like simulate_parallel but streams results back as chunks complete so tqdm
    can update after every `games_per_chunk` games rather than only at the end.
    Requires joblib >= 1.2 for return_as="generator".
    """
    n_chunks = max(n_jobs, -(-n_games // games_per_chunk))  # ceiling division
    base = n_games // n_chunks
    extras = n_games % n_chunks
    chunk_sizes = [base + (1 if i < extras else 0) for i in range(n_chunks)]
    seeds = [random.randint(0, 2**31) for _ in range(n_chunks)]

    all_scores = []
    with tqdm(total=n_games, desc=strategy_name, unit="game") as pbar:
        for chunk_scores in Parallel(n_jobs=n_jobs, backend="loky", return_as="generator")(
            delayed(_run_chunk)(strategy_name, c, s)
            for c, s in zip(chunk_sizes, seeds)
        ):
            all_scores.extend(chunk_scores)
            pbar.update(len(chunk_scores))

    return all_scores

In [ ]:
N = 500_000
N_EV = 500  # EV is expensive; 500 games with 16 cores ≈ 2-3 min

results = {}

for display_name, key, n in [
    ("Random",         "random",         N),
    ("Greedy",         "greedy",         N),
    ("Yahtzee Hunter", "yahtzee_hunter", N),
]:
    t0 = time.perf_counter()
    results[display_name] = simulate_parallel(key, n)
    dt = time.perf_counter() - t0
    print(f"{display_name:<18} {n:>6,} games  |  {dt:>5.1f}s  |  mean = {np.mean(results[display_name]):.1f}")

print()
t0 = time.perf_counter()
# games_per_chunk=1 so the bar ticks after every single game (~3-5s each)
results["EV"] = simulate_parallel_progress("ev", N_EV, games_per_chunk=1)
dt = time.perf_counter() - t0
print(f"{'EV':<18} {N_EV:>6,} games  |  {dt:>5.1f}s  |  mean = {np.mean(results['EV']):.1f}")

Random             500,000 games  |    6.4s  |  mean = 45.9
Greedy             500,000 games  |   12.8s  |  mean = 134.6
Yahtzee Hunter     500,000 games  |   13.2s  |  mean = 140.0



ev:   0%|          | 0/500 [00:00<?, ?game/s]

## Results

In [ ]:
print(f"{'Strategy':<18} {'Mean':>7} {'Median':>7} {'Std':>7} {'Min':>6} {'Max':>6}")
print("-" * 55)
for name, scores in results.items():
    a = np.array(scores)
    print(f"{name:<18} {a.mean():>7.1f} {np.median(a):>7.1f} {a.std():>7.1f} {a.min():>6} {a.max():>6}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distributions
ax = axes[0]
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]
for (name, scores), color in zip(results.items(), colors):
    ax.hist(scores, bins=40, alpha=0.6, label=name, color=color, density=True)
ax.set_xlabel("Final Score")
ax.set_ylabel("Density")
ax.set_title("Score Distributions by Strategy")
ax.legend()

# Mean comparison bar chart
ax2 = axes[1]
names = list(results.keys())
means = [np.mean(s) for s in results.values()]
bars = ax2.bar(names, means, color=colors, edgecolor="black", alpha=0.85)
ax2.set_ylabel("Mean Final Score")
ax2.set_title("Average Score by Strategy")
for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{val:.1f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylim(0, max(means) * 1.15)

plt.tight_layout()
plt.show()

## Validation: Smart Keeps vs Exhaustive (All 32)
Tests whether the smart pruning heuristic meaningfully changes EV strategy quality.
Run with n=200 each. If p > 0.05 and the difference is within the confidence interval, pruning is validated.

In [ ]:
from scipy import stats

# Exhaustive variant — identical to strategy_ev but uses all 32 keeps
def _all_keep_sets(dice):
    for r in range(6):
        for combo in combinations(range(5), r):
            yield list(combo)

def expected_score_exhaustive(keep_indices, dice, scorecard, remaining_rolls, n_samples=EV_SAMPLES):
    open_cats = open_categories(scorecard)
    if not open_cats:
        return 0
    total = 0
    for _ in range(n_samples):
        sim_dice = reroll(dice, keep_indices)
        if remaining_rolls > 1:
            best_val = 0
            for sub_keep in _all_keep_sets(sim_dice):
                sub_ev = expected_score_exhaustive(
                    sub_keep, sim_dice, scorecard, remaining_rolls - 1,
                    n_samples=EV_SUB_SAMPLES
                )
                best_val = max(best_val, sub_ev)
            total += best_val
        else:
            total += max(score_dice(sim_dice, c) for c in open_cats)
    return total / n_samples

def strategy_ev_exhaustive(dice, roll_num, scorecard):
    open_cats = open_categories(scorecard)
    remaining_rolls = 3 - roll_num
    if remaining_rolls == 0:
        cat = max(open_cats, key=lambda c: score_dice(dice, c))
        return list(range(5)), cat
    best_keep, best_val = list(range(5)), -1
    for keep in _all_keep_sets(dice):
        ev = expected_score_exhaustive(keep, dice, scorecard, remaining_rolls)
        if ev > best_val:
            best_val = ev
            best_keep = keep
    return best_keep, None

STRATEGY_MAP["ev_exhaustive"] = strategy_ev_exhaustive

N_VAL = 200

print("Smart keeps EV (n=200):")
t0 = time.perf_counter()
val_smart = simulate_parallel_progress("ev", N_VAL, games_per_chunk=1)
print(f"Done in {time.perf_counter()-t0:.0f}s\n")

print("Exhaustive EV (n=200):")
t0 = time.perf_counter()
val_exhaustive = simulate_parallel_progress("ev_exhaustive", N_VAL, games_per_chunk=1)
print(f"Done in {time.perf_counter()-t0:.0f}s\n")

# Two-sample t-test
t_stat, p_val = stats.ttest_ind(val_smart, val_exhaustive)
mean_diff = np.mean(val_smart) - np.mean(val_exhaustive)
se_diff = np.sqrt(np.var(val_smart, ddof=1)/N_VAL + np.var(val_exhaustive, ddof=1)/N_VAL)
ci = 1.96 * se_diff

print(f"Smart keeps:  mean = {np.mean(val_smart):.1f}  std = {np.std(val_smart):.1f}")
print(f"Exhaustive:   mean = {np.mean(val_exhaustive):.1f}  std = {np.std(val_exhaustive):.1f}")
print(f"Difference:   {mean_diff:+.1f} ± {ci:.1f} pts  (95% CI)")
print(f"p = {p_val:.3f}  →  {'no significant difference ✓' if p_val > 0.05 else 'SIGNIFICANT difference — pruning may be lossy'}")